# DATASCI 350 - Data Science Computing

## Assignment 09 - Parallel computing and SQL

### Instructions

This assignment evaluates your understanding of lectures 21 and 22: serial and parallel execution, `map`, `joblib`, Big O notation, Amdahl's law, the GIL, Dask, storage formats, and SQL with DuckDB.

Most tasks are reading and reasoning about material you already have. You read a snippet of code, a set of timings, or a table, and you explain what it means.

Eight tasks ask you to run code: tasks 04, 05, 06, 08, 09, 10, 11 and 12. Install what they need with `pip install joblib numpy duckdb pandas dask`. Tasks 04, 05 and 06 use the `%timeit` magic, so run them inside Jupyter or IPython rather than in a plain `python` script.

Tasks 08 to 12 use one file, `wdi_panel.parquet`, which is 450 KB. Download it once with the `curl` line in task 08 and reuse it for the other four. Nothing else in this assignment downloads anything.

Your timings will not match mine, and they do not need to. Core counts, memory and background processes differ between machines. Report what your machine did, paste the actual output, and answer the questions about the numbers you got. Do not edit outputs.

You must complete this assignment individually. You may use notes, books, and AI tools, but you must submit original work. I strongly recommend that you do not use AI tools to generate your solutions, as this will not help you learn the material. Acknowledge all resources used, including input from classmates and AI, in a short note at the end of your submission. If you are unsure about permissible resources or proper acknowledgement, please ask the instructor.

### Submission

Please submit your solutions as either a single Jupyter notebook or a PDF file. For each task, paste the commands you ran, any code you wrote, the output you got, and your written answer where the task asks for one. Screenshots are welcome but not required if you paste the text. Submit your completed assignment to Canvas.

### Task 01

Here are three jobs.

1. Resizing 400 holiday photographs to thumbnails. Each photograph is a separate file and none of them depends on another.
2. Computing a running total over a column of 400 numbers, where each entry is the previous total plus the next number.
3. Fetching 200 web pages and saving the HTML of each one.

For each job, say whether it is embarrassingly parallel, whether it is the kind of job threads help with, or whether neither applies. One line per job is enough.

Then answer in a few sentences: pick the job that does not benefit from either approach and say what about it blocks parallel work.

Hint: the "Serial vs parallel execution" slide of lecture 21 defines embarrassingly parallel, and the `concurrent.futures` slide separates CPU-bound from I/O-bound work.

*Commands, output, and answer here.*

### Task 02

Amdahl's law says the speed-up you can get from `N` cores is

$$\text{speedup} = \frac{1}{s + \frac{p}{N}}$$

where `s` is the share of the program that must run serially and `p = 1 - s` is the share that can be split across cores.

You have profiled a program and found that a quarter of its runtime is serial: `s = 0.25`, so `p = 0.75`.

Fill in this table. Show the arithmetic for each row, not just the final number, and round the speed-up to two decimal places.

| Cores | Speed-up |
|-------|----------|
| 4 | |
| 8 | |
| 32 | |
| infinitely many | |

Then answer in a few sentences: which of the two jumps buys you the least extra speed, 4 to 8 cores or 8 to 32 cores? Given the ceiling you computed in the last row, say what you would work on instead of buying more cores.

Hint: the "Amdahl's law: the speed-up ceiling" slide of lecture 21 works through the same arithmetic with `s = 0.1` on 8 cores.

*Commands, output, and answer here.*

### Task 03

A classmate wrote a Python function that loops over ten million numbers and adds them up in pure Python, with no NumPy. They ran it with eight threads and reported that it was no faster than the single-threaded version, and slightly slower on some runs.

Answer in a few sentences. First, explain why the threads did not help, using the GIL. Second, their code uses `ThreadPoolExecutor`, and one word in that name has to change to fix the problem. Name the replacement and say why it works.

Then answer this separately, in one or two sentences: the same classmate has a second script that downloads 500 web pages. Threads would help there. Say what is different about the second job.

Hint: the "The GIL" slide of lecture 21, including the table on the right.

*Commands, output, and answer here.*

### Task 04

This task times the same work three ways on your own machine. Run it in Jupyter or IPython, because `%timeit` is a notebook magic and does not work in a plain script.

Start by reporting your core count:

```python
import os
print(os.cpu_count())
```

Then define the function from the "Serial vs parallel execution" slide of lecture 21:

```python
import numpy as np
from joblib import Parallel, delayed

def calculation(size=10000000):
    # Create a large array and perform operations
    arr = np.random.rand(size)
    for _ in range(10):
        arr = np.sqrt(arr) + np.sin(arr)
    return np.mean(arr)
```

Now time three things, one cell each, and paste all three results:

```python
%timeit calculation()
```

```python
%timeit [calculation() for _ in range(4)]
```

```python
%timeit Parallel(n_jobs=4)(delayed(calculation)() for _ in range(4))
```

Then answer two things.

1. Divide your serial time for four calls by your parallel time for four calls, and report that number as your speed-up. Show the division.
2. In a few sentences, say why the speed-up is less than 4 even though you asked for four workers, and name two things joblib has to do that the serial version does not.

Your numbers will differ from mine and from your classmates'. That is expected, and you are graded on the reasoning, not the milliseconds.

Hint: the "Serial vs parallel execution" slide of lecture 21 runs exactly these three timings and the note at the bottom right names the two costs.

*Commands, output, and answer here.*

### Task 05

Task 04 showed joblib winning. This task finds the point where it starts to win. Run it in Jupyter or IPython, because `%timeit` is a notebook magic.

Here are three functions doing the same arithmetic at three different costs.

```python
import numpy as np
from joblib import Parallel, delayed

def square(x):
    return x**2

def square_1000(x):
    for _ in range(1000):
        result = x**2
    return result

def square_100000(x):
    for _ in range(100000):
        result = x**2
    return result

numbers = np.arange(10000)
```

Time each function twice, once serially and once in parallel, for six timings in all. The first pair looks like this:

```python
%timeit [square(x) for x in numbers]
```

```python
%timeit Parallel(n_jobs=4)(delayed(square)(x) for x in numbers)
```

Do the same two lines for `square_1000`. The pair for `square_100000` is slow, about half a minute for the serial run on my machine, so add `-n1 -r1` to make `%timeit` take one measurement instead of seven:

```python
%timeit -n1 -r1 [square_100000(x) for x in numbers]
```

```python
%timeit -n1 -r1 Parallel(n_jobs=4)(delayed(square_100000)(x) for x in numbers)
```

Paste all six timings.

Then answer three things.

1. Name the first of the three functions where the parallel version beat the serial one.
2. In a few sentences, say what that tells you about the fixed cost of handing one call to a worker process. Use your own numbers. There are 10,000 calls in every one of the six timings, so the only thing changing is how much work one call does.
3. State the rule you would use to decide whether joblib is worth reaching for. One or two sentences.

Hint: Exercise 01 and Appendix 01 of lecture 21 run the losing case, with `square` and nothing heavier. This task asks where the losing stops.

*Commands, output, and answer here.*

### Task 06

This task builds a three-stage pipeline with `dask.delayed` and compares it with the same pipeline run serially. Run it in Jupyter or IPython.

Here are the three functions from the "Dask delayed" slides of lecture 21.

```python
import numpy as np
import dask

def generate_data(size):
    return np.random.rand(size)

def transform_data(data):
    return np.sqrt(data) + np.sin(data)

def aggregate_data(data):
    return {
        'mean': np.mean(data),
        'std': np.std(data),
        'max': np.max(data)
    }
```

First version. Put `@dask.delayed` above each of the three functions, chain them over three input sizes, and compute everything in one call.

```python
sizes = [1000000, 2000000, 3000000]

results = []
for size in sizes:
    data = generate_data(size)
    transformed = transform_data(data)
    stats = aggregate_data(transformed)
    results.append(stats)

%timeit dask.compute(*results)
```

Second version. The same three functions without the decorator, called in a plain `for` loop over the same three sizes, appending each `aggregate_data` result to a list. Time that with `%timeit` too.

Paste both timings.

Then answer two things.

1. The pipeline is nine tasks. How many of them can run at the same moment, and why? Say what stops the rest from starting.
2. Suppose `aggregate_data` took all three transformed arrays at once and returned one set of statistics for the lot. Say what that would change about the shape of the graph, and about your answer to question 1.

Hint: the "A decorator schedules any function you already have" and "Dask draws the plan before running a single task" slides of lecture 21, both titled "Dask delayed".

*Commands, output, and answer here.*

### Task 07

Lecture 21 wrote one month of generated data twice, once as 30 CSV files and once as a single Parquet file, then ran the same group-by on each. The numbers were:

| | CSV | Parquet |
|---|-----|---------|
| Size on disk | 182 MB | 86 MB |
| Group-by time | 738 ms | 73.6 ms |

Answer in a few sentences, treating the two gaps separately.

1. The size gap: name the two properties of Parquet that make the same data smaller on disk.
2. The speed gap: the group-by touched two columns out of the several in the file. Explain why that matters for Parquet and not for CSV. Say whether the file being smaller is enough on its own to explain a tenfold speed-up.

Then name one situation where you would still hand somebody a CSV, and say why Parquet would be the wrong choice there.

Hint: the "Reading and writing data" and "Why Parquet?" slides of lecture 21.

*Commands, output, and answer here.*

### Task 08

This task and the three after it use the WDI panel from lecture 19: 59,024 rows of country, ISO3 code, indicator, year and value. Download it once, into whatever folder you are working in:

```bash
curl -O https://raw.githubusercontent.com/danilofreire/datasci350/main/lectures/lecture-19/data/wdi_panel.parquet
```

The file is about 450 KB. Keep it next to your notebook so the paths below work as written.

Now run these four cells and paste the output of each.

```python
import duckdb

duckdb.sql("SELECT * FROM 'wdi_panel.parquet' LIMIT 5")
```

```python
duckdb.sql("""
    CREATE OR REPLACE VIEW wdi AS
    SELECT * FROM 'wdi_panel.parquet'
""")
```

```python
duckdb.sql("DESCRIBE SELECT * FROM wdi")
```

```python
duckdb.sql("""
    SELECT column_name, min, max, null_percentage
    FROM (SUMMARIZE SELECT * FROM wdi)
""")
```

The second cell prints nothing. That is correct: creating a view returns no rows.

Then answer in a few sentences: read the `null_percentage` column and say which of the five columns has missing data and roughly how much. Then say what that means for a query that computes `AVG(value)`, given that SQL aggregates skip missing values without telling you.

Hint: the "Querying a Parquet file", "Creating a view" and "DESCRIBE and SUMMARIZE" slides of lecture 22.

*Commands, output, and answer here.*

### Task 09

Keep the `wdi` view from task 08. Write one query that answers: which countries have had the highest life expectancy since 2010?

Your query must do all of this:

1. Keep only the rows where `indicator` is `life_expectancy`.
2. Keep only the years from 2010 onwards.
3. Compute the mean of `value` for each country.
4. Keep only the countries whose mean is above 82.
5. Round the mean to one decimal place.
6. Sort from highest to lowest.
7. Return the top five.

Paste the query and the result table.

Then answer in one or two sentences: step 2 and step 4 are both filters, but they go in different clauses. Name the clause each one belongs to and say why step 4 cannot go where step 2 goes.

Hint: Appendix 02 of lecture 22 asks this exact question, and the "WHERE and HAVING" slide explains the split between the two clauses.

*Commands, output, and answer here.*

### Task 10

Keep the `wdi` view. This question is about how many rich countries there were in each recent year.

Write one query that, for the `gdp_per_capita` indicator:

1. Counts how many countries had a value above 30,000 in each year from 2010 to 2023.
2. Keeps only the years where that count is above 40.
3. Returns the year and the count, ordered by year.

Paste the query and the result table. You should get fewer than fourteen rows: at least one year does not clear the threshold.

Then answer in a few sentences. Two of the conditions you wrote are filters: "value above 30,000" and "count above 40". Say which clause you put each one in, and explain why they cannot be swapped.

Hint: the "WHERE and HAVING" slide of lecture 22 uses `WHERE` on a row condition and `HAVING` on an aggregate in the same query. The "COUNT(\*) and COUNT(value)" slide shows counting inside a `GROUP BY`.

*Commands, output, and answer here.*

### Task 11

Keep the `wdi` view. Write one query about the `internet_users_pct` indicator in 2020, for these six countries: Brazil, Japan, India, Germany, Nigeria and the United States. The country name in the panel is `United States`.

The result must have four columns:

1. `country`.
2. The value, rounded to one decimal place.
3. The mean value across the six countries in your result, on every row. Compute it with `AVG(value) OVER ()`, and round it to one decimal place too.
4. A column called `position` that says `above` when the country's value is at or above that mean and `below` otherwise.

Note the empty brackets in `AVG(value) OVER ()`. The slide writes `OVER (PARTITION BY year)`, which makes one window per year. Empty brackets mean one window covering the whole result, so the number you get is the mean of the rows your `WHERE` clause kept, not the mean of all 217 countries.

Paste the query and the result.

Then answer in one or two sentences: `GROUP BY country` with `AVG(value)` would also give you a mean. Say what you would lose if you used it here instead of the window.

Hint: the "Window functions" slide of lecture 22 for `OVER`, and the "CASE" slide for the if-else ladder inside a `SELECT`.

*Commands, output, and answer here.*

### Task 12

This task joins two small tables. Keep the `wdi` view from task 08, then run these four statements from the "Two small tables" slide of lecture 22.

```python
duckdb.sql("""
    CREATE OR REPLACE TABLE gdp2020 AS
    SELECT country, iso3, ROUND(value, 0) AS gdp_per_capita
    FROM wdi
    WHERE indicator = 'gdp_per_capita' AND year = 2020
      AND iso3 IN ('USA','BRA','CHN','IND','DEU','JPN')
""")
```

```python
duckdb.sql("""
    CREATE OR REPLACE TABLE regions
        (iso3 VARCHAR PRIMARY KEY, region VARCHAR)
""")
```

```python
duckdb.sql("""
    INSERT INTO regions VALUES
        ('USA', 'North America'), ('BRA', 'Latin America'),
        ('CHN', 'East Asia'),     ('IND', 'South Asia'),
        ('DEU', 'Europe'),        ('KOR', 'East Asia')
""")
```

```python
duckdb.sql("SELECT * FROM regions ORDER BY iso3")
```

Six rows on each side, with two mismatches: `JPN` has a GDP figure and no region, `KOR` a region and no GDP figure.

Now add two more rows to `regions`, for a region with no country in `gdp2020` at all.

```python
duckdb.sql("""
    INSERT INTO regions VALUES
        ('EGY', 'North Africa'), ('MAR', 'North Africa')
""")
```

Neither `EGY` nor `MAR` is one of the six codes in `gdp2020`, so North Africa is now a region with no GDP data behind it.

Write one query returning one row per region, with three columns: the region, how many of that region's countries have a 2020 GDP figure, and the mean GDP per capita of the region rounded to the nearest whole number. Every region must appear, North Africa included, so join with a `LEFT JOIN`. Sort by mean GDP, highest first, and send North Africa to the bottom with `NULLS LAST`.

Paste the query and the result. You should get six rows, the last of them North Africa with a count of 0 and an empty mean.

Then answer in a few sentences: say why this needs a `LEFT JOIN` rather than an `INNER JOIN`, and which of the two tables has to be the one named in `FROM`. Say what the result would look like if you swapped them.

Hint: the "LEFT JOIN" slide of lecture 22, and the exercise at the end of the joins section. Appendix 01 gives the solution to that exercise, and it is the same shape as the query here, the extra region being the only difference.

*Commands, output, and answer here.*